# What defines the neighbourhoods of proliferating and transport-losing tubules? (notebook 110)

**The question.** In the IR mice, cell-cycle (CYC) and transport (TPT) residuals cluster among neighbouring tubules
beyond position and depth (notebook 96), and both programs are patchy while injury markers are near-universal
(notebook 107). Is there a repair niche, and what is in it?

**The frozen protocol** is `docs/results/pt-repair-niches.md` (commit 5e3df08).
- D1: does neighbours' injury predict a tubule's CYC (and TPT loss) beyond its own injury, position and depth?
- D2: which interstitial lineage in the tubule's peritubular pool (fibroblast, myeloid, neutrophil, endothelium) or
  its own hypoxia response marks the niche?
- D3: does D1 survive when only non-touching neighbours (1.25–3 diameters) are used?
- Controls: CYC on interior bins (C1), E2F against G2M (C2), timing (C3).

**Sections.** 1 setup; 2 peritubular pools (cached); 3 scores and neighbours; 4 D1; 5 D2; 6 D3; 7 controls;
8 verdicts; 9 figures; 10 reading and run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import inspect
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
PKG = results_root / 'package_pseudospace'
EXPORT = PKG / 'aki' / 'pt_units.h5ad'
RUN01 = PKG / 'runs' / 'aki' / '01_units'
T96 = results_root / 'pt_injury_tissue_location' / 'tables'
T76 = results_root / 'pt_package_aki' / 'tables'
T104 = results_root / 'pt_contamination_audit' / 'tables'
output = results_root / 'pt_repair_niches'
tables, figures, cache_root = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '110.repair_niches.1'
SEED = 110
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
SPECIMENS = CONTROLS + IRS
KS, PRIMARY_KS = (3, 6, 12, 24), (3, 6)
TOUCH, FAR = 1.25, 3.0
EDGE_MAX, RING_MAX, CORE = 1, 3.0, 3
N_PERM, ALPHA, SD_FLOOR, BIN_WIDTH = 2500, .05, .25, .05
LINEAGES = {
    'fibroblast': ['Col1a1', 'Col1a2', 'Col3a1', 'Dcn', 'Pdgfrb', 'Pdgfra', 'Postn', 'Timp1', 'Tnc', 'Acta2', 'Crlf1', 'Npr3', 'Igfbp5'],
    'myeloid': ['C1qa', 'C1qb', 'C1qc', 'Cd68', 'Lyz2', 'Csf1r', 'Itgam', 'Adgre1', 'Cd74', 'Mrc1', 'Fcgr1', 'Ms4a7'],
    'neutrophil': ['S100a8', 'S100a9', 'Csf3r', 'Cxcr2', 'Mmp8', 'Retnlg'],
    'endothelium': ['Pecam1', 'Emcn', 'Plvap', 'Kdr', 'Flt1'],
}
HYPOXIA = ['Egln3', 'Bnip3', 'Ndrg1', 'Adm', 'Slc2a1', 'Ldha', 'Pdk1', 'Ankrd37', 'P4ha1', 'Vegfa']
for path in (EXPORT, RUN01 / 'manifest.json', T96 / 'units.csv', T76 / 'gene_sets.json', T76 / 'universe.csv',
             T104 / 'edge_enrichment.csv.gz', *[RUN01 / f'{s}_bin_to_unit.parquet' for s in SPECIMENS]):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import sparse
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from IPython.display import display

from pseudospace import contamination as ct
from pseudospace.package_inputs import read_pt_units, support_positions
from pseudospace.regulator_activity import stratified_permutations
from pseudospace.stage_cache import cached_payload
from pseudospace.tissue_location import knn_weights, morans_i, position_expectation, program_score

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'IR2A2': '#D55E00', 'IR2A4': '#E69F00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
units = read_pt_units(EXPORT, 'aki')
obs = units.obs
mask, position = support_positions(obs.pseudospace.to_numpy(float), obs['sample'].astype(str).to_numpy(), obs.segment_class)
GENES = pd.Index(units.var_names.astype(str))
COUNTS = sparse.csr_matrix(units.X[mask])
U = pd.DataFrame({'specimen': obs['sample'].astype(str).to_numpy()[mask], 'x': position,
                  'polygon': obs.polygon_id.astype(str).to_numpy()[mask],
                  'diameter': obs.equivalent_diameter_um.to_numpy(float)[mask]}, index=obs.index[mask].astype(str))
del units
U96 = pd.read_csv(T96 / 'units.csv', index_col=0)
U96.index = U96.index.astype(str)
assert U96.index.equals(U.index), 'G1: notebook 96 units differ from the export common support'
for c in ('cx', 'cy', 'depth pct', 'INJ', 'TPT', 'CYC', 'e INJ', 'e TPT', 'e CYC'):
    U[c] = U96[c].to_numpy()
guard_log.append(f'G1: 14,048 common-support units with notebook 96\'s scores, residuals, depth and centroids ({len(U):,})')

sets76 = json.loads((T76 / 'gene_sets.json').read_text())
u76 = set(pd.read_csv(T76 / 'universe.csv').gene.astype(str))
G2M = sorted(set(sets76['MSigDB_Hallmark_2020::G2-M Checkpoint']) & u76)
E2F = sorted(set(sets76['MSigDB_Hallmark_2020::E2F Targets']) & u76)
CYC = sorted(set(G2M) | set(E2F))
assert len(CYC) == 297, 'G2: CYC as notebook 96'
LIN = {k: [g for g in v if g in GENES] for k, v in LINEAGES.items()}
HYP = [g for g in HYPOXIA if g in GENES]
print('lineage markers in the export:', {k: len(v) for k, v in LIN.items()}, '· hypoxia', len(HYP))
guard_log.append('G2: lineage markers present ' + ', '.join(f'{k} {len(v)}/{len(LINEAGES[k])}' for k, v in LIN.items())
                 + f'; hypoxia {len(HYP)}/{len(HYPOXIA)}')
MANIFEST = json.loads((RUN01 / 'manifest.json').read_text())
UM_PER_PX = {}

## 2 · Peritubular pools and core aggregates (cached per specimen)

In [ ]:
def read_bins(folder):
    if (folder / 'filtered_feature_bc_matrix.h5').is_file():
        with h5py.File(folder / 'filtered_feature_bc_matrix.h5', 'r') as f:
            m = f['matrix']
            X = sparse.csc_matrix((m['data'][:], m['indices'][:], m['indptr'][:]), shape=tuple(m['shape'][:])).T.tocsr()
            return X, m['barcodes'][:].astype(str)
    from scipy.io import mmread
    mtx = folder / 'filtered_feature_bc_matrix'
    return sparse.csr_matrix(mmread(mtx / 'matrix.mtx.gz').T), pd.read_csv(mtx / 'barcodes.tsv.gz', header=None)[0].astype(str).to_numpy()


def pools(spec):
    names = list(MANIFEST['samples'])
    folder = Path(MANIFEST['inputs'][2 * names.index(spec)])
    X, barcodes = read_bins(folder)
    u1 = ad.read_h5ad(RUN01 / f'{spec}_units.h5ad', backed='r')
    cols = pd.Index(u1.var_names.astype(str)).get_indexer(GENES)
    del u1
    X = X[:, cols].tocsr()
    rows, colsb = ct.bin_grid(barcodes)
    b2u = pd.read_parquet(RUN01 / f'{spec}_bin_to_unit.parquet')
    polygons = pd.Index(b2u.polygon_id.astype(str).unique())
    label = np.full(len(barcodes), -1)
    label[pd.Index(barcodes).get_indexer(b2u.bin_id.astype(str))] = polygons.get_indexer(b2u.polygon_id.astype(str))
    depth = ct.label_depth(rows, colsb, label)
    ring = ct.peritubular_ring(rows, colsb, label, max_dist=RING_MAX)
    here = U.specimen.eq(spec).to_numpy()
    pt_poly = polygons.get_indexer(U.polygon.to_numpy()[here])
    lut = np.full(len(polygons), -1)
    lut[pt_poly] = np.arange(len(pt_poly))
    own = np.where(label >= 0, lut[label.clip(0)], -1)
    ringu = np.where(ring >= 0, lut[ring.clip(0)], -1)
    n = len(pt_poly)
    peri = np.where((own >= 0) & (depth <= EDGE_MAX), own, ringu)          # edge bins of the unit + its outside ring
    pool = ct.unit_aggregate(X, peri, n)
    core = ct.unit_aggregate(X, own, n, keep=depth >= CORE)
    lin_cols = {k: GENES.get_indexer(v) for k, v in LIN.items()}
    cyc_cols = GENES.get_indexer(CYC)
    return {'pool_total': np.asarray(pool.sum(axis=1)).ravel(),
            **{f'pool|{k}': np.asarray(pool[:, c].sum(axis=1)).ravel() for k, c in lin_cols.items()},
            'ring_bins': np.bincount(ringu[ringu >= 0], minlength=n),
            'core_total': np.asarray(core.sum(axis=1)).ravel(),
            'core_cyc': core[:, cyc_cols].toarray()}


POOLS = {}
for spec in SPECIMENS:
    t0 = time.perf_counter()
    POOLS[spec] = cached_payload(f'pools_{spec}', lambda: pools(spec), root=cache_root,
                                 params={'logic': NOTEBOOK_LOGIC_VERSION, 'edge': EDGE_MAX, 'ring': RING_MAX, 'core': CORE,
                                         'lineages': LIN},
                                 inputs={'export': str(EXPORT), 'specimen': spec}, code=inspect.getsource(pools) + inspect.getsource(ct))
    names = list(MANIFEST['samples'])
    sf = json.loads((Path(MANIFEST['inputs'][2 * names.index(spec)]) / 'spatial' / 'scalefactors_json.json').read_text()) \
        if (Path(MANIFEST['inputs'][2 * names.index(spec)]) / 'spatial' / 'scalefactors_json.json').is_file() else None
    UM_PER_PX[spec] = float(sf['microns_per_pixel']) if sf else np.nan
    print(f'{spec}: {len(POOLS[spec]["pool_total"]):,} units; median peritubular counts {np.median(POOLS[spec]["pool_total"]):.0f}; '
          f'median ring bins {np.median(POOLS[spec]["ring_bins"]):.0f} ({time.perf_counter() - t0:.0f} s)')
if any(np.isnan(v) for v in UM_PER_PX.values()):
    fallback = float(np.nanmedian(list(UM_PER_PX.values())))
    UM_PER_PX = {k: (v if np.isfinite(v) else fallback) for k, v in UM_PER_PX.items()}
print('µm per pixel:', {k: round(v, 4) for k, v in UM_PER_PX.items()})

## 3 · Scores, residuals and neighbours

In [ ]:
lib = np.asarray(COUNTS.sum(axis=1)).ravel()


def expression(genes):
    block = COUNTS[:, GENES.get_indexer(genes)].toarray()
    return np.log1p(block / lib[:, None] * 1e4)


ref = U.specimen.isin(CONTROLS).to_numpy()


def residual(score):
    return score - position_expectation(U.x, score, U.specimen, CONTROLS, width=BIN_WIDTH, cross_fit=True)


for name, genes in (('G2M', G2M), ('E2F', E2F), ('HYP', HYP)):
    U[name] = program_score(expression(genes), ref, floor=SD_FLOOR)
    U[f'e {name}'] = residual(U[name].to_numpy())
# lineage scores and core CYC, in U order
for k in LIN:
    vals = np.concatenate([np.log1p(1e4 * POOLS[s][f'pool|{k}'] / np.maximum(POOLS[s]['pool_total'], 1)) for s in SPECIMENS])
    order = np.concatenate([np.flatnonzero(U.specimen.eq(s).to_numpy()) for s in SPECIMENS])
    col = np.empty(len(U)); col[order] = vals
    mu, sd = col[ref].mean(), max(col[ref].std(ddof=1), 1e-9)
    U[f'L {k}'] = (col - mu) / sd
order = np.concatenate([np.flatnonzero(U.specimen.eq(s).to_numpy()) for s in SPECIMENS])
core_cyc = np.vstack([POOLS[s]['core_cyc'] for s in SPECIMENS])
core_tot = np.concatenate([POOLS[s]['core_total'] for s in SPECIMENS])
core_expr = np.empty_like(core_cyc, dtype=float)
core_expr[order] = np.log1p(core_cyc / np.maximum(core_tot, 1)[:, None] * 1e4)
U['CYC core'] = program_score(core_expr, ref, floor=SD_FLOOR)
U['e CYC core'] = residual(U['CYC core'].to_numpy())
ring_bins = np.empty(len(U)); ring_bins[order] = np.concatenate([POOLS[s]['ring_bins'] for s in SPECIMENS])
U['ring bins'] = ring_bins

# neighbours
for k in KS:
    U[f'nbr INJ k{k}'] = np.nan
U['nbr INJ far'] = np.nan
U['touching share k6'] = np.nan
for s in SPECIMENS:
    m = U.specimen.eq(s).to_numpy()
    xy = U.loc[m, ['cx', 'cy']].to_numpy() * UM_PER_PX[s]
    d, idx = cKDTree(xy).query(xy, k=max(KS) + 1)
    d, idx = d[:, 1:], idx[:, 1:]
    e = U.loc[m, 'e INJ'].to_numpy()
    diam = U.loc[m, 'diameter'].to_numpy()
    pair_diam = (diam[:, None] + diam[idx]) / 2
    for k in KS:
        U.loc[m, f'nbr INJ k{k}'] = e[idx[:, :k]].mean(axis=1)
    far = (d >= TOUCH * pair_diam) & (d <= FAR * pair_diam)
    with np.errstate(invalid='ignore'):
        U.loc[m, 'nbr INJ far'] = np.where(far.any(axis=1), (e[idx] * far).sum(axis=1) / far.sum(axis=1), np.nan)
    U.loc[m, 'touching share k6'] = (d[:, :6] < TOUCH * pair_diam[:, :6]).mean(axis=1)
U['stratum'] = -1
for s in SPECIMENS:
    m = U.specimen.eq(s)
    xd = pd.qcut(U.loc[m, 'x'].rank(method='first'), 10, labels=False)
    dt = pd.qcut(U.loc[m, 'depth pct'].rank(method='first'), 3, labels=False)
    U.loc[m, 'stratum'] = (xd * 3 + dt).to_numpy()
U.to_csv(tables / 'units.csv')
display(U.groupby('specimen')[['e INJ', 'e CYC', 'e TPT', 'e G2M', 'e E2F', 'e HYP', *[f'L {k}' for k in LIN], 'touching share k6', 'ring bins']].mean().round(3))

## 4 · D1: neighbours' injury against a tubule's proliferation and transport

In [ ]:
def fwl_coef(y, x, Z):
    """Coefficient of x in y ~ x + Z (+ intercept), by residualising on Z."""
    Z1 = np.column_stack([np.ones(len(y)), Z])
    beta_y = np.linalg.lstsq(Z1, y, rcond=None)[0]
    ry = y - Z1 @ beta_y
    beta_x = np.linalg.lstsq(Z1, x, rcond=None)[0]
    rx = x - Z1 @ beta_x
    return float(rx @ ry / (rx @ rx)), Z1


def perm_test(y, x, Z, strata, seed, *, sign=1):
    coef, Z1 = fwl_coef(y, x, Z)
    Q, _ = np.linalg.qr(Z1)
    ry = y - Q @ (Q.T @ y)
    perms = stratified_permutations(strata, N_PERM, seed)
    Xp = x[perms].T                                          # n x perms
    Rx = Xp - Q @ (Q.T @ Xp)
    null = (Rx * ry[:, None]).sum(axis=0) / (Rx ** 2).sum(axis=0)
    p = (1 + (sign * null >= sign * coef).sum()) / (N_PERM + 1)
    return coef, p, float(np.quantile(null, .025)), float(np.quantile(null, .975))


def zscore(v):
    v = np.asarray(v, float)
    return (v - np.nanmean(v)) / np.nanstd(v)


def d1_rows(outcome, sign, predictor_cols):
    rows = []
    for s in SPECIMENS:
        f = U[U.specimen.eq(s)].dropna(subset=[outcome, *predictor_cols, 'e INJ'])
        own = zscore(f['e INJ'])
        Z = np.column_stack([own, own ** 2, zscore(f['depth pct'])])
        for col in predictor_cols:
            coef, p, lo, hi = perm_test(zscore(f[outcome]), zscore(f[col]), Z, f.stratum.to_numpy(), SEED + len(rows), sign=sign)
            rows.append({'outcome': outcome, 'predictor': col, 'specimen': s, 'units': len(f), 'coef': coef,
                         'p (one-sided)': p, 'null 2.5%': lo, 'null 97.5%': hi})
    return pd.DataFrame(rows)


NBR = [f'nbr INJ k{k}' for k in KS]
D1 = pd.concat([d1_rows('e CYC', 1, NBR), d1_rows('e TPT', -1, NBR)], ignore_index=True)
D1.to_csv(tables / 'd1.csv', index=False)
display(D1.pivot_table(index=['outcome', 'predictor'], columns='specimen', values='coef').round(3))
display(D1.pivot_table(index=['outcome', 'predictor'], columns='specimen', values='p (one-sided)').round(4))


def holds(table, outcome, predictor, sign):
    t = table[(table.outcome == outcome) & (table.predictor == predictor)].set_index('specimen')
    ir_ok = all(sign * t.loc[s, 'coef'] > 0 and t.loc[s, 'p (one-sided)'] <= ALPHA for s in IRS)
    beyond = all(sign * t.loc[i, 'coef'] > sign * t.loc[c, 'coef'] for i in IRS for c in CONTROLS)
    return bool(ir_ok and beyond)


D1_VERDICT = {(o, k): holds(D1, o, f'nbr INJ k{k}', sg) for o, sg in (('e CYC', 1), ('e TPT', -1)) for k in KS}
D1_HOLDS = {o: all(D1_VERDICT[(o, k)] for k in PRIMARY_KS) for o in ('e CYC', 'e TPT')}
print('D1 per k:', D1_VERDICT, '\nD1 holds (k = 3 and 6):', D1_HOLDS)

## 5 · D2: which lineage marks the niche?

In [ ]:
PRED2 = [f'L {k}' for k in LIN] + ['e HYP']


def d2_rows(outcome):
    rows = []
    for s in SPECIMENS:
        f = U[U.specimen.eq(s)].dropna(subset=[outcome, *PRED2, 'e INJ'])
        own = zscore(f['e INJ'])
        base = [own, own ** 2, zscore(f['depth pct'])]
        for j, col in enumerate(PRED2):
            others = [zscore(f[c]) for c in PRED2 if c != col]
            Z = np.column_stack(base + others)
            coef, p_pos, lo, hi = perm_test(zscore(f[outcome]), zscore(f[col]), Z, f.stratum.to_numpy(), SEED + 100 + j)
            p_neg = perm_test(zscore(f[outcome]), zscore(f[col]), Z, f.stratum.to_numpy(), SEED + 100 + j, sign=-1)[1]
            rows.append({'outcome': outcome, 'predictor': col, 'specimen': s, 'units': len(f), 'coef': coef,
                         'p (> 0)': p_pos, 'p (< 0)': p_neg, 'null 2.5%': lo, 'null 97.5%': hi})
    return pd.DataFrame(rows)


D2 = pd.concat([d2_rows('e CYC'), d2_rows('e TPT')], ignore_index=True)
D2.to_csv(tables / 'd2.csv', index=False)
display(D2.pivot_table(index=['outcome', 'predictor'], columns='specimen', values='coef').round(3))
display(D2.pivot_table(index=['outcome', 'predictor'], columns='specimen', values='p (> 0)').round(4))
cyc2 = D2[D2.outcome.eq('e CYC')]
fib_ok = all(((cyc2.specimen == s) & (cyc2.predictor == 'L fibroblast') & (cyc2.coef > 0) & (cyc2['p (> 0)'] <= ALPHA)).any() for s in IRS)
fib_top = all(cyc2[(cyc2.specimen == s) & cyc2.predictor.str.startswith('L ')].sort_values('coef', ascending=False).predictor.iloc[0] == 'L fibroblast'
              for s in IRS)
D2_FIBRO = bool(fib_ok and fib_top)
print('D2 fibroblast niche holds:', D2_FIBRO)

## 6 · D3: beyond touching neighbours

In [ ]:
D3 = d1_rows('e CYC', 1, ['nbr INJ far'])
D3_HOLDS = all(((D3.specimen == s) & (D3.coef > 0) & (D3['p (one-sided)'] <= ALPHA)).any() for s in IRS)
moran = []
for s in SPECIMENS:
    f = U[U.specimen.eq(s)]
    xy = f[['cx', 'cy']].to_numpy() * UM_PER_PX[s]
    W = knn_weights(xy, k=6).tocoo()
    d = np.linalg.norm(xy[W.row] - xy[W.col], axis=1)
    pd_ = (f.diameter.to_numpy()[W.row] + f.diameter.to_numpy()[W.col]) / 2
    keep = d >= TOUCH * pd_
    Wf = sparse.csr_matrix((np.ones(keep.sum()), (W.row[keep], W.col[keep])), shape=W.shape)
    rs = np.asarray(Wf.sum(axis=1)).ravel()
    Wf = sparse.diags(np.divide(1, rs, out=np.zeros_like(rs, dtype=float), where=rs > 0)) @ Wf
    for col in ('e CYC', 'e TPT'):
        moran.append({'specimen': s, 'program': col, 'I all pairs': morans_i(f[col].to_numpy(), W.tocsr()),
                      'I non-touching': morans_i(f[col].to_numpy(), Wf), 'touching share': float(1 - keep.mean())})
MORAN = pd.DataFrame(moran)
pd.concat([D3.assign(kind='D3 non-touching neighbour INJ'), MORAN.assign(kind='Moran')], ignore_index=True).to_csv(tables / 'd3.csv', index=False)
display(D3.round(4)); display(MORAN.round(3))
print('D3 beyond touching holds:', D3_HOLDS)

## 7 · Controls

In [ ]:
eps = pd.read_csv(T104 / 'edge_enrichment.csv.gz', index_col=0)
aki_cols = [c for c in eps.columns if c.startswith('aki ')]
C1_EPS = eps.reindex(CYC)[aki_cols].median().to_dict()
C1_RHO = {s: float(spearmanr(U.loc[U.specimen.eq(s), 'e CYC'], U.loc[U.specimen.eq(s), 'e CYC core'], nan_policy='omit').statistic) for s in SPECIMENS}
D1_CORE = d1_rows('e CYC core', 1, [f'nbr INJ k{k}' for k in PRIMARY_KS])
D1_HALF = pd.concat([d1_rows(f'e {h}', 1, [f'nbr INJ k{k}' for k in PRIMARY_KS]) for h in ('G2M', 'E2F')], ignore_index=True)
edges = np.linspace(0, 1, 11)
U['xbin'] = np.clip(np.digitize(U.x, edges[1:-1]), 0, 9)
HALVES = U.groupby(['specimen', 'xbin'])[['e G2M', 'e E2F']].mean()
det = {}
for g in ('Mki67', 'Top2a'):
    if g in GENES:
        col = np.asarray((COUNTS[:, GENES.get_loc(g)] > 0).toarray()).ravel()
        det[g] = pd.Series(col, index=U.index).groupby(U.specimen).mean().to_dict()
CONTROLS_TABLE = pd.concat([
    pd.DataFrame({'control': 'C1 median edge enrichment of CYC genes', 'key': list(C1_EPS), 'value': list(C1_EPS.values())}),
    pd.DataFrame({'control': 'C1 ρ(e CYC, e CYC core)', 'key': list(C1_RHO), 'value': list(C1_RHO.values())}),
    D1_CORE.assign(control='C1 D1 on core CYC').rename(columns={'specimen': 'key', 'coef': 'value'})[['control', 'key', 'value', 'predictor', 'p (one-sided)']],
    D1_HALF.assign(control='C2 D1 on each half').rename(columns={'specimen': 'key', 'coef': 'value'})[['control', 'key', 'value', 'predictor', 'outcome', 'p (one-sided)']],
    pd.DataFrame([{'control': f'C3 share of units detecting {g}', 'key': s, 'value': v} for g, d in det.items() for s, v in d.items()]),
], ignore_index=True)
CONTROLS_TABLE.to_csv(tables / 'controls.csv', index=False)
display(CONTROLS_TABLE.round(4))
display(HALVES.unstack('xbin').round(2))

## 8 · Verdicts

In [ ]:
def coef_text(table, outcome, predictor):
    t = table[(table.outcome == outcome) & (table.predictor == predictor)].set_index('specimen')
    return '; '.join(f"{s} {t.loc[s, 'coef']:+.3f} (p {t.loc[s, [c for c in t.columns if c.startswith('p ')][0]]:.3g})" for s in SPECIMENS)


VERDICTS = pd.DataFrame([
    ('D1 neighbour injury → own CYC (k 3 and 6)', f"{D1_HOLDS['e CYC']} · k3: {coef_text(D1, 'e CYC', 'nbr INJ k3')}"),
    ('D1 neighbour injury → own TPT loss (k 3 and 6)', f"{D1_HOLDS['e TPT']} · k3: {coef_text(D1, 'e TPT', 'nbr INJ k3')}"),
    ('D2 fibroblast niche for CYC', str(D2_FIBRO)),
    *[(f'D2 {p} → CYC', coef_text(D2, 'e CYC', p)) for p in PRED2],
    *[(f'D2 {p} → TPT', coef_text(D2, 'e TPT', p)) for p in PRED2],
    ('D3 beyond touching (non-touching neighbour INJ → CYC)', f"{D3_HOLDS} · {coef_text(D3, 'e CYC', 'nbr INJ far')}"),
], columns=['reading', 'result'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 9 · Figures (plots only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(110 * MM, 42 * MM), sharey=False)
for ax, outcome in zip(axes, ('e CYC', 'e TPT')):
    t = D1[D1.outcome.eq(outcome)]
    for s in SPECIMENS:
        ts = t[t.specimen.eq(s)]
        ks = [int(p.split('k')[-1]) for p in ts.predictor]
        ax.plot(ks, ts.coef, marker='o', ms=2.5, color=SPEC_COLOR[s])
        ax.fill_between(ks, ts['null 2.5%'], ts['null 97.5%'], color=SPEC_COLOR[s], alpha=.12, lw=0)
    ax.axhline(0, color=LIGHT, lw=.5)
    ax.set_xscale('log', base=2); ax.set_xticks(list(KS), [str(k) for k in KS])
    ax.set_xlabel('neighbours (k)'); ax.set_title(outcome.replace('e ', ''))
axes[0].set_ylabel('neighbour-injury coefficient')
save(fig, '110a_d1_by_k')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(110 * MM, 42 * MM), sharey=True)
for ax, outcome in zip(axes, ('e CYC', 'e TPT')):
    t = D2[D2.outcome.eq(outcome)]
    yy = np.arange(len(PRED2))
    for off, s in zip((-.27, -.09, .09, .27), SPECIMENS):
        ts = t[t.specimen.eq(s)].set_index('predictor').reindex(PRED2)
        ax.scatter(ts.coef, yy + off, color=SPEC_COLOR[s], s=8)
    ax.axvline(0, color=LIGHT, lw=.5)
    ax.set_yticks(yy, [p.replace('L ', '').replace('e HYP', 'hypoxia') for p in PRED2])
    ax.set_title(outcome.replace('e ', ''))
save(fig, '110b_d2_lineages')
plt.show()

fig, ax = plt.subplots(figsize=(70 * MM, 42 * MM))
for i, (s, g) in enumerate(MORAN.groupby('specimen', sort=False)):
    for j, (_, r) in enumerate(g.iterrows()):
        ax.plot([i * 3 + j, i * 3 + j], [r['I non-touching'], r['I all pairs']], color=SPEC_COLOR[s])
        ax.scatter([i * 3 + j], [r['I all pairs']], color=SPEC_COLOR[s], s=8)
        ax.scatter([i * 3 + j], [r['I non-touching']], facecolor='white', edgecolor=SPEC_COLOR[s], s=8)
ax.set_xticks([i * 3 + .5 for i in range(4)], SPECIMENS)
ax.set_ylabel("Moran's I (k 6)")
save(fig, '110c_moran_touching')
plt.show()

f = U[U.specimen.eq('IR2A2')]
fig, axes = plt.subplots(1, 2, figsize=(120 * MM, 55 * MM))
for ax, col, cmap in zip(axes, ('e CYC', 'L fibroblast'), ('RdBu_r', 'viridis')):
    lo, hi = np.nanquantile(f[col], [.02, .98])
    ax.scatter(f.cx, -f.cy, c=f[col], s=1, cmap=cmap, vmin=lo, vmax=hi, lw=0)
    ax.set_aspect('equal'); ax.axis('off')
save(fig, '110d_map_ir2a2')
plt.show()

## 10 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'Residuals of G2M, E2F, hypoxia and core CYC use notebook 96\'s recipe: program_score on log1p CP10k standardised on control units (SD floor 0.25), minus position_expectation (width 0.05, cross-fitted).'),
    ('reading', 'Predictors and outcomes are z-scored within each mouse; the permutation null moves the predictor of interest within position-decile × depth-tercile strata of that mouse, other covariates fixed (Frisch–Waugh).'),
    ('reading', 'D2 fits all lineage scores and hypoxia together; each coefficient is tested with the others as covariates.'),
    ('reading', 'Touching pairs use the mean of the two units\' equivalent diameters; µm per pixel come from each specimen\'s scale factors.'),
    ('reading', 'The peritubular pool is the unit\'s own edge bins (depth ≤ 1) plus outside bins whose nearest polygon within 3 bins is the unit.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '110_pt_repair_niches', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'um_per_px': UM_PER_PX, 'd1_verdict': {f'{o} k{k}': v for (o, k), v in D1_VERDICT.items()}, 'd1_holds': D1_HOLDS,
       'd2_fibroblast': D2_FIBRO, 'd3_holds': D3_HOLDS, 'moran': MORAN.round(4).to_dict('records'),
       'c1_rho': C1_RHO, 'c1_eps': C1_EPS, 'c3_detection': det, 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('d1_holds', 'd2_fibroblast', 'd3_holds', 'seconds')}, indent=2, default=str))